# Cookbook: point-source MT (MTUQ) + finite-fault (WISP) for any earthquake
**Self-service, step-by-step.** Set the parameters in the next cell, then run cells top to bottom.
Each step calls the battle-tested scripts in `scripts/` (all lessons from nb19–nb24 baked in:
displacement tags, `water_level=None`, syngine needs data ≥2 Hz, contamination checks, PZ rename,
true BH1/BH2 azimuths, DC-constraint for shallow events, depth search, SNR-bypass for marginal P,
surface-waves-only for deep events, WISP GF-bank depth cap).
Worked examples: 2026 Sanriku-oki M6.9 interplate (nb22, contamination) and
2026 Calama M6.9 intraslab at 109 km (nb24, depth).

**Environments:** `mtuq` (MTUQ+MPI; `export FI_PROVIDER=tcp`), `ff-env` (WISP/ffm CLI), `tex`
(tectonic, for the report). Magnitude guidance: MT works to M~5.5+; FFI needs Mw ≳ 7.0 for real
slip detail (6.7–7.0 marginal; below 6.5 do MT only) — **except deep events**, whose impulsive P
and in-window depth phases let WISP work at M6.9 (nb24).

**Depth guidance (nb24):** hypocentre ≳ 60–70 km ⇒ (a) MTUQ body band mixes P/pP/sP and can
converge ~90° wrong — run DC searches with `--nobody` (surface waves only); (b) WISP's
surface-wave GF bank ends at ~126 km — keep the fault's deepest row above it or WISP silently
drops all surface waves for that plane.

In [1]:
# ============ 1. EVENT PARAMETERS (single source of truth) ============
EVENT_ID = "us6000t7zq"            # USGS ComCat ID
TAG      = "2026sanriku"           # short tag for data/results dirs
TIME     = "2026-06-24T22:30:12.99"
LAT, LON, DEPTH_KM, MAG = 40.2515, 142.1614, 34.0, 6.9
EVENT_DIR = f"event_{TAG}"         # WISP working dir (created below)
# Optional: another large event nearby in time whose wavetrain may contaminate windows
CONTAM = dict(time_offset_s=-1502., lat=10.435, lon=-68.472)   # None to skip
import os, subprocess, json
BASE = os.path.expanduser('~/works/17.Venezuela_2026'); os.chdir(BASE)
def run(cmd, env=None):
    print('$', cmd)
    e = f"source ~/miniforge3/etc/profile.d/conda.sh && conda activate {env} && " if env else ""
    r = subprocess.run(['bash','-lc', e+cmd], capture_output=True, text=True)
    print(r.stdout[-1500:]); print(r.stderr[-500:] if r.returncode else '')
    return r

## 2. USGS metadata — confirm mechanism, depth, and whether an FFM already exists

In [ ]:
meta_py = '''
import json
d = json.load(open("/tmp/ev.json")); p = d["properties"]
print(p["title"]); print("FFM exists?", "finite-fault" in p["products"])
for mt in p["products"].get("moment-tensor", []):
    q = mt["properties"]
    print(q.get("derived-magnitude-type"), "depth", q.get("derived-depth"),
          q.get("nodal-plane-1-strike"), q.get("nodal-plane-1-dip"), q.get("nodal-plane-1-rake"))
'''
open('/tmp/ev_meta.py', 'w').write(meta_py)
r = run(f"curl -s 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid={EVENT_ID}&format=geojson' "
        "-o /tmp/ev.json && python3 /tmp/ev_meta.py")

## 3. MTUQ — data prep (teleseismic GSN 3-C, displacement, no water level)
`13_prep_mtuq_2009.py` is fully parameterized. Keep `--sr 5` (syngine rejects dt=1.0!).
If you ever resample yourself: `obspy interpolate(method='lanczos', a=20)` — the default
`weighted_average_slopes` is NOT band-limited. And MTUQ's syngine resampler has an
npts-dependent off-by-one ("could not broadcast (N,) into (N−1,)"): if it fires, trim the
traces to a working sample count (12800 @ 4 Hz is known-good) rather than fighting it.

In [ ]:
run(f"python -u scripts/13_prep_mtuq_2009.py --event {TAG} --time {TIME} "
    f"--lat {LAT} --lon {LON} --depth {DEPTH_KM} --sr 5 --nmax 26 --mindeg 25 --maxdeg 75", env='mtuq')

## 4. Contamination check (optional but recommended)
If a larger event occurred within ~2 h, compute per-station window overlaps and zero the CAP
weights of swept channels (columns after offset = bodyZ bodyR surfZ surfR surfT). See nb22 §A for
the full recipe; template below prints the overlap table so YOU decide which stations to zero.

In [ ]:
if CONTAM:
    script = f'''
import glob
from obspy.io.sac import SACTrace
from obspy.geodetics.base import gps2dist_azimuth
from obspy.taup import TauPyModel
m=TauPyModel("ak135")
for f in sorted(glob.glob("data/mtuq/{TAG}/*.z")):
    s=SACTrace.read(f,headonly=True)
    dV=gps2dist_azimuth({CONTAM["lat"]},{CONTAM["lon"]},s.stla,s.stlo)[0]/1000.
    tp=m.get_travel_times({DEPTH_KM},s.dist/111.19,["P"])[0].time
    v=({CONTAM["time_offset_s"]}+dV/4.7, {CONTAM["time_offset_s"]}+dV/2.9)
    sw=(s.dist/4.6, s.dist/3.2)
    print(s.kstnm, "P-hit" if (tp-15)<v[1] and v[0]<(tp+65) else "  ok ",
          "SW-hit" if sw[0]<v[1] and v[0]<sw[1] else "  ok")
'''
    open('/tmp/contam.py','w').write(script); run("python /tmp/contam.py", env='mtuq')

## 5. MTUQ inversions — full-MT, DC, DC+depth (MPI ×20), then VR
For shallow events (<25 km) prefer the DC result (shallow-source indeterminacy, nb21).

**For deep events (≳60–70 km) the DEFAULT 10–33 s body band is actively misleading** (nb24):
the 40-s window mixes P/pP/sP, and with source duration and depth-phase smear comparable to
the pP−P lag there is NO short-period band where a point source is valid (validity needs
T ≫ duration & smear; resolving pP as a distinct arrival needs T ≲ pP−P lag). Two working
options, validated on Calama (M6.9 @109 km):
- `--nobody` — surface waves only (safe; loses body information AND depth resolution);
- **`--bwband 25,60,120,10` (preferred)** — move the body band to 25–60 s with a 120-s window
  holding the whole P+pP+sP group and ±10 s shifts, GFs at the depth-phase-constrained
  centroid. On Calama this flipped body VR from −80 %/−24 % (ours/USGS) to **+71.9 %/+67.7 %**,
  killed the spurious flipped mechanism (18.5 %), and the joint body+SW DC reached **Kagan
  8.4°** from USGS Mww (vs 17.7 % SW-only) beating the reference on both bands. Bonus: the
  long-period body waves retain *interference-shape* depth sensitivity (W-phase-like, even
  with pP unresolved) — a body-only `16_run --bwband ... --onlybody` depth scan gave a genuine
  interior minimum (117 km), where the SW-only scan was monotonic/unresolved.

A strongly non-DC full-MT (|w| near 1) at depth is the short-band failure showing up as a red
flag, not a discovery.

In [ ]:
pre = "export FI_PROVIDER=tcp I_MPI_FABRICS=shm:ofi && "
run(pre+f"mpirun -n 20 python -u scripts/12_run_mtuq_sota.py --event {TAG} --lat {LAT} "
    f"--lon {LON} --depth {DEPTH_KM} --time '{TIME}' --mag {MAG} --npts 10", env='mtuq')

In [ ]:
nb_flag = " --nobody" if DEPTH_KM >= 60 else ""     # deep event: surface waves only (nb24)
run(pre+f"mpirun -n 20 python -u scripts/15_run_mtuq_dc.py --event {TAG} --lat {LAT} "
    f"--lon {LON} --depth {DEPTH_KM} --time '{TIME}' --mag {MAG}{nb_flag}", env='mtuq')

In [ ]:
depths = ",".join(str(int(DEPTH_KM+d)) for d in (-8,-4,0,4,8,12))
nb_flag = " --nobody" if DEPTH_KM >= 60 else ""     # deep event: surface waves only (nb24)
run(pre+f"mpirun -n 20 python -u scripts/16_run_mtuq_dc_depth.py --event {TAG} --lat {LAT} "
    f"--lon {LON} --time '{TIME}' --mag {MAG} --depths {depths}{nb_flag}", env='mtuq')
# If the best depth lands on the EDGE of the list, extend the list and rerun — an edge
# minimum is not a minimum. And check the misfit-vs-depth CURVE, not just the argmin:
# nb24's SW-only 55-115 km search was MONOTONIC (no interior minimum) — long-period surface
# waves constrain the mechanism of a deep source but NOT its centroid depth; adopt depth from
# depth phases (WISP / published Mwc) and say so. Trust plot_misfit_depth/origin_idxmin —
# MTUQ's misfit array is (sources, origins), origins LAST (a wrong-axis reshape once printed
# a fake interior minimum; fixed in 16_run).

Inspect: `results/mtuq_sota_<TAG>/`, `mtuq_dc_<TAG>/`, `mtuq_dcz_<TAG>/` — beachball, lune, DC misfit surfaces, misfit-vs-depth, waveform fits. Compare mechanisms via Kagan angle (nb21 §B has the implementation), never plane-by-plane angles.

## 6. WISP finite-fault — CMT, data, header fix, auto model (both planes)
The three-step pipeline is `scripts/run_wisp_sanriku.sh` — copy it for your event (edit dir+CMT):
1. `ffm get-data cmt <ID> -d .` then `ffm get-data teleseismic <ID> -d .` (tens of minutes; if it
   stalls on unresponsive networks with 100+ BHZ already down, kill it and proceed).
   **If teleseismic discovery fails outright** (`FDSNNoServiceException` — the packaged IRIS
   `http://` endpoint is dead), use `scripts/wisp_fetch_fallback.py <time> <lat> <lon>
   [mindeg maxdeg]` — an obspy replacement that writes the same `<NET>_<STA>_<CHA>_<LOC>.sac`
   + `SAC_PZs_*` layout (nb24 used it for all 180 channels).
2. strip `.sac` from `SAC_PZs_*` names; `python scripts/wisp_fix_headers.py . <CMT>` (coords +
   TRUE BH1/BH2 azimuths — never assume 1→N/2→E)
3. `ffm model run . auto_model -g <CMT> -d . -t body -t surf`

**After the run, grep `wisp_pipeline.log` for "Maximum depth larger than 125"**: if the fault
plane bottoms below WISP's ~126-km surface-wave GF bank, WISP silently drops ALL surface waves
for that plane — its misfit is then body-only and NOT comparable to the other plane's (nb24:
this manufactured a fake 0.114-vs-0.166 plane discrimination). Fix: reduce `dip_subfaults` in
`segments_data.json` until the deepest row is above the cap, and refine both planes identically.

In [ ]:
os.makedirs(EVENT_DIR, exist_ok=True)
run(f"cd {EVENT_DIR} && ffm get-data cmt {EVENT_ID} -d .", env='ff-env')
print('Now run (long):  cd', EVENT_DIR, f'&& ffm get-data teleseismic {EVENT_ID} -d .')
print('Then adapt scripts/run_wisp_sanriku.sh steps 1-3 for this directory.')

## 7. WISP refinement — QC + shift-match (+ P force-include if selection rejects P)
Adapt `scripts/rerun_calama_refine.py` (bad-response + optional contamination QC + shift_match +
re-invert; parameterized by plane: `python scripts/rerun_calama_refine.py NP1`). Gotchas:
tele_waves.json must stay ordered BHZ-block-then-BHT; `save_waveforms` after shift_match can drop
the P block — re-merge it (see nb22 story); at M<7 WISP's SNR gate may reject ALL P — bypass with
`dm.select_tele_stations = lambda f,phase,ti: f` on a curated clean-station list, using the
`data/P/processed_*` files (never the intermediate SACs — Fortran buffer overflow). Deep events
(nb24) don't need the bypass: impulsive deep P passes SNR≥5 easily.

**Refine BOTH planes with identical treatment before claiming plane discrimination.** Compare
misfits only when both planes invert the same wave types and channel set (see the GF-cap trap in
§6). For a compact rupture the conjugates are usually indistinguishable — say so, present one by
tectonic convention, and verify both planes' slip-weighted mechanisms via Kagan angle
(`pyrocko.moment_tensor.kagan_angle` — use the canonical implementation, hand-rolled Kagan code
fails on conjugate representations).

In [ ]:
print('Figures: scripts/fig_wisp_slipdist_5s.py <NP_dir>   (WISP style, 5-s isochrones)')
print('         scripts/fig_fault_isochrone.py <NP_dir>    (true-scale slip+vectors)')

## 8. Interpretation rubric (hard-won)
- Robust at any magnitude: Mw, mechanism family, centroid depth (if depth phases in band).
- FFI extent/directivity: trust only if source duration ≥ 3× shortest well-modeled period (~8 s
  teleseismic) → Mw ≳ 7.0. Below: report patch location/size as soft, choose plane by tectonics.
  Deep events relax this (impulsive P + depth phases), but the conjugate-plane ambiguity stays.
- Fit metrics: report VR per wave type on CLEAN stations; radiation-node stations legitimately
  show flat synthetics; body-amplitude deficits can be slab-path/site effects, not source.
- Always compare to GCMT/USGS via Kagan angle — with `pyrocko.moment_tensor.kagan_angle`, not a
  hand-rolled version (conjugate representations break naive T/B/P sign-combination code);
  verify conjugate planes numerically (Kagan ≈ 0).
- Evaluate the *reference* mechanism's VR on your own data (nb20's water-level bug and nb24's
  body-window flip were both caught this way): if the published solution fits your data terribly,
  suspect your prep or your band before claiming a discovery.
- A misfit/depth/plane minimum on the EDGE of a searched grid is not a minimum — extend and rerun.